In [8]:
import colorama
from colorama import Fore, Style
from IPython.display import display

def preview(file, sheet_name, n=5):
    print(Fore.BLUE+Style.BRIGHT+f"\n* Preview of '{sheet_name}' ({file.split('/')[-1]}) *\n")
    
    df = pd.read_excel(file, sheet_name=sheet_name)

    # Select numeric columns
    numeric_cols = df.select_dtypes(include="number").columns

    if len(numeric_cols) > 0:
        # Keep only rows that contain at least one numeric value (not all NaN)
        df = df[df[numeric_cols].notna().any(axis=1)]

    # Show first n rows
    df_styled = df.head(n).style.set_properties(**{'text-align': 'left'})
    df_styled = df_styled.set_table_styles(
        [{'selector': 'th', 'props': [('text-align', 'left')]}]
    )

    display(df_styled)


In [10]:
import pandas as pd

# Specify the path to the Excel file
file_path = '/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels.xlsx'
# load the template created by Script 01
xl = pd.ExcelFile(file_path)

try:
    input_tech_existing = pd.read_excel(file_path, sheet_name='Input-Technology')
    output_tech_existing = pd.read_excel(file_path, sheet_name='Output-Technology')
except Exception as e:
    print(f"Error reading existing sheets: {e}")
    # Initialize with 'Technology' and 'Sector' columns if sheets are not found
    input_tech_existing = pd.DataFrame(columns=['Technology', 'Sector'])
    output_tech_existing = pd.DataFrame(columns=['Technology', 'Sector'])

years = ['2025', '2030', '2035', '2040', '2045', '2050']
for year in years:
    df_year = xl.parse(year, header=0)

    # Process each year's data
    input_df = df_year[df_year['Energy input/output'] > 0].groupby('Technology', as_index=False)['Energy input/output'].sum()
    output_df = df_year[df_year['Energy input/output'] < 0].groupby('Technology', as_index=False)['Energy input/output'].sum()

    # Rename the aggregated column to the year
    input_df.rename(columns={'Energy input/output': year}, inplace=True)
    output_df.rename(columns={'Energy input/output': year}, inplace=True)

    # Update or add the year column for both input and output
    for df_new, df_key in zip([input_df, output_df], ['input', 'output']):
        # Determine which DataFrame to update
        df_existing = input_tech_existing if df_key == 'input' else output_tech_existing
        
        # Merge and directly update the original DataFrame
        merged_df = pd.merge(df_existing, df_new, on='Technology', how='outer', suffixes=('', '_new')).fillna(0)
        # Handle the potential new column
        if f'{year}_new' in merged_df.columns:
            merged_df[year] = merged_df[f'{year}_new']
            merged_df.drop(columns=f'{year}_new', inplace=True)
        
        # Update the original DataFrame
        if df_key == 'input':
            input_tech_existing = merged_df
        else:
            output_tech_existing = merged_df

# Reorder 'Sector' column if necessary
for df in [input_tech_existing, output_tech_existing]:
    if 'Sector' in df.columns:
        sector_col = df.pop('Sector')
        df.insert(1, 'Sector', sector_col)

# Write updated data back to the Excel file, replacing old sheets
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    input_tech_existing.to_excel(writer, sheet_name='Input-Technology', index=False)
    output_tech_existing.to_excel(writer, sheet_name='Output-Technology', index=False)

print("Aggregation and updating Excel file completed.")


Aggregation and updating Excel file completed.


In [12]:
preview(file_path, "Input-Technology")
preview(file_path, "Output-Technology")


* Preview of 'Input-Technology' (Book-EnergyBal with biofuels.xlsx) *



,Technology,Sector,2025,2030,2035,2040,2045,2050
0,1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw),Huishoudens,0.000000,0.000000,0.000000,0.000000,0.000001,0.000001
1,1010 Micro-WKK Brandstofcel 2011 (WB bestaande bouw),Huishoudens,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1012 Solar-PV Residential,Huishoudens,37.080362,29.859767,30.326279,47.025141,31.259398,48.450141
3,1013 Biomassavergassing met CCS 7000u 2011,Elektriciteitsopwekking,0.000000,0.000054,0.000021,0.000000,0.000005,0.000014
4,1016 ATR with CC,H2 voorziening,0.000000,0.000414,13.702892,26.707571,17.447790,15.973219



* Preview of 'Output-Technology' (Book-EnergyBal with biofuels.xlsx) *



,Technology,Sector,2025,2030,2035,2040,2045,2050
0,1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw),Huishoudens,0.000000,-0.000001,-0.000001,0.000000,-0.000001,-0.000001
1,1010 Micro-WKK Brandstofcel 2011 (WB bestaande bouw),Huishoudens,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1012 Solar-PV Residential,Huishoudens,-37.080362,-29.859767,-30.326279,-47.025141,-31.259398,-48.450141
3,1013 Biomassavergassing met CCS 7000u 2011,Elektriciteitsopwekking,0.000000,-0.000006,-0.000003,0.000000,-0.000002,-0.000005
4,1016 ATR with CC,H2 voorziening,0.000000,-0.000358,-11.647310,-22.701145,-14.830432,-13.577064


In [14]:
#Read the "Output-Technology" sheet

xl = pd.ExcelFile(file_path)
output_tech_df = pd.read_excel(xl, 'Output-Technology')

# Summing up energy values for each sector across the years

sector_summary = output_tech_df.groupby('Sector').sum(numeric_only=True).reset_index()

# Write the aggregated data to a new sheet in the Excel file
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    sector_summary.to_excel(writer, sheet_name='Energy Output-Sector', index=False)

print("Aggregated energy per sector summary has been added to the Excel file.")

input_tech_df=pd.read_excel(xl, 'Input-Technology')

sectorin_summary = input_tech_df.groupby('Sector').sum(numeric_only=True).reset_index()
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    sectorin_summary.to_excel(writer, sheet_name='Energy Input-Sector', index=False)

Aggregated energy per sector summary has been added to the Excel file.


In [15]:
preview(file_path, "Energy Output-Sector")
preview(file_path, "Energy Input-Sector")


* Preview of 'Energy Output-Sector' (Book-EnergyBal with biofuels.xlsx) *



,Sector,2025,2030,2035,2040,2045,2050
0,0,-0.000001,0.000000,0.000000,0.000000,0.000000,0.000000
1,Basic metal non-ferro,-1.682099,-1.459087,-1.403072,-1.452084,-1.438083,-1.438084
2,Basismetaal ferro,-33.936316,-23.708988,-18.918488,-17.463773,-18.314740,-18.739960
3,Biofuel,-61.958938,-32.599296,-25.865194,-64.238391,-84.626822,-116.371707
4,Bunkers,-0.000001,0.000000,0.000000,0.000000,0.000000,0.000000



* Preview of 'Energy Input-Sector' (Book-EnergyBal with biofuels.xlsx) *



,Sector,2025,2030,2035,2040,2045,2050
0,0,337.910803,0.000000,0.000000,0.000000,0.000000,0.000000
1,Basic metal non-ferro,3.366872,2.932078,2.814625,2.918013,2.882320,2.882749
2,Basismetaal ferro,148.262628,117.840441,105.414333,100.018966,98.706572,96.577824
3,Biofuel,72.600779,35.764080,28.157273,75.902717,104.197273,152.984498
4,Bunkers,256.204377,567.090215,553.880041,540.570010,524.770004,510.600085


In [16]:

xl = pd.ExcelFile(file_path)

sector_mapping = pd.read_excel(xl, 'Input-Technology')[['Technology', 'Sector']].drop_duplicates()

years = ['2025','2030', '2035', '2040', '2045', '2050']
all_data = []

# Step 1 & 2: Read yearly data and merge with sector information
for year in years:
    df_year = xl.parse(year)
    df_year_with_sector = pd.merge(df_year, sector_mapping, on='Technology', how='left')
    df_year_with_sector['Year'] = year  # Add a year column for concatenation
    all_data.append(df_year_with_sector[['Energy Carrier', 'Energy input/output', 'Sector', 'Year']])

# Combine yearly data into a single DataFrame
df_combined = pd.concat(all_data)

# Step 3: Aggregate Energy Input and Output by Energy Carrier and Sector
# Aggregate data, specifying numeric_only=True to avoid FutureWarning
input_per_carrier_sector = df_combined[df_combined['Energy input/output'] > 0].groupby(['Energy Carrier', 'Sector', 'Year']).sum(numeric_only=True).reset_index()
output_per_carrier_sector = df_combined[df_combined['Energy input/output'] < 0].groupby(['Energy Carrier', 'Sector', 'Year']).sum(numeric_only=True).reset_index()

# Pivot table to transform aggregated data, each year's data in its own column
inputcarr_summary = input_per_carrier_sector.pivot_table(index=['Energy Carrier', 'Sector'], columns='Year', values='Energy input/output', fill_value=0).reset_index()
outputcarr_summary = output_per_carrier_sector.pivot_table(index=['Energy Carrier', 'Sector'], columns='Year', values='Energy input/output', fill_value=0).reset_index()

# Writing the summaries to new Excel sheets
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    inputcarr_summary.to_excel(writer, sheet_name='Energy input per carrier', index=False)
    outputcarr_summary.to_excel(writer, sheet_name='Energy Output per carrier', index=False)
    
print("Done.")

Done.


In [20]:
preview(file_path, "Energy input per carrier")
preview(file_path, "Energy Output per carrier")


* Preview of 'Energy input per carrier' (Book-EnergyBal with biofuels.xlsx) *



,Energy Carrier,Sector,2025,2030,2035,2040,2045,2050
0,Aardgas,Basic metal non-ferro,0.559743,0.705912,0.629872,0.702697,0.315822,0.337394
1,Aardgas,Basismetaal ferro,0.000356,21.091734,32.333775,32.095020,25.791173,23.144778
2,Aardgas,Chemie,43.938362,33.964527,26.775830,19.115965,10.645055,9.178334
3,Aardgas,Elektriciteitsopwekking,330.000001,164.543035,96.549437,66.209741,67.686415,70.298685
4,Aardgas,Food and Beverage Industry,62.511431,19.648805,7.172284,5.403230,3.650708,20.225698



* Preview of 'Energy Output per carrier' (Book-EnergyBal with biofuels.xlsx) *



,Energy Carrier,Sector,2025,2030,2035,2040,2045,2050
0,Aardgas,Basismetaal ferro,0.000000,0.000000,-0.140997,0.000000,0.000000,0.000000
1,Aardgas,Gasvoorziening,-4.311471,-0.000248,-0.000101,-0.000015,-0.000037,-0.000196
2,Aardgas,HDO,0.000000,-0.000845,-0.000270,-0.000457,-3.095944,-2.058743
3,Aardgas,Huishoudens,0.000000,-0.000195,-0.000071,-0.000366,-2.411602,-1.776896
4,Aardgas,Landbouw,-0.000000,-8.799965,-14.999909,-14.999999,0.000000,-0.000000


In [22]:

xl = pd.ExcelFile(file_path)

years = ['2025','2030', '2035', '2040', '2045', '2050']
all_data = []

for year in years:
    df_year = xl.parse(year, header=0)
    df_year['Year'] = year  # Add a year column
    all_data.append(df_year[['Technology', 'Energy Carrier', 'Energy input/output', 'Year']])

# Combine all yearly data into a single DataFrame
combined_df = pd.concat(all_data)

# Pivot the data
pivoted_df = combined_df.pivot_table(index=['Technology', 'Energy Carrier'],
                                     columns='Year',
                                     values='Energy input/output',
                                     aggfunc='sum').reset_index()

# Reset the index to turn the multi-index into columns
pivoted_df.reset_index(inplace=True)
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    pivoted_df.to_excel(writer, sheet_name='Tech & Carrier by Year', index=False)

print("Sheet with technology, energy carrier, and energy input/output for each year has been added.")
sector_mapping_df = pd.read_excel(file_path, sheet_name='Input-Technology', usecols=['Technology', 'Sector']).drop_duplicates()
# Merge sector information into the combined data
combined_df_with_sector = pd.merge(combined_df, sector_mapping_df, on='Technology', how='left')
# Pivot the data with Sector included
pivoted_df_with_sector = combined_df_with_sector.pivot_table(index=['Technology', 'Energy Carrier', 'Sector'],
                                                             columns='Year',
                                                             values='Energy input/output',
                                                             aggfunc='sum').reset_index()

# Ensure the index is correctly reset if needed (though reset_index might not be necessary after pivot_table depending on your pandas version)
pivoted_df_with_sector.reset_index(drop=True, inplace=True)

# Write the updated DataFrame to a new sheet
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    pivoted_df_with_sector.to_excel(writer, sheet_name='Tech, Carrier & Sector by Year', index=False)
    
with pd.ExcelWriter('/Users/ahmedelberry/Library/CloudStorage/OneDrive-UvA/GEM-E3/Linking/Book-EnergyBal with biofuels_inv.xlsx',
                    engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    pivoted_df_with_sector.to_excel(writer, sheet_name='Tech, Carrier & Sector by Year', index=False)

print("Sheet with technology, energy carrier, sector, and energy input/output for each year has been added.")


Sheet with technology, energy carrier, and energy input/output for each year has been added.
Sheet with technology, energy carrier, sector, and energy input/output for each year has been added.


In [93]:
preview(file_path, "Tech & Carrier by Year")
preview(file_path, "Tech, Carrier & Sector by Year")


* Preview of 'Tech & Carrier by Year' (Book-EnergyBal with biofuels.xlsx) *



,index,Technology,Energy Carrier,2025,2030,2035,2040,2045,2050
0,0,1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw),Elektriciteit,0.000000,-0.000000,-0.000000,0.000000,-0.000000,-0.000000
1,1,1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw),Warmte,0.000000,-0.000000,-0.000000,0.000000,-0.000001,-0.000001
2,2,1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw),Waterstof,0.000000,0.000000,0.000000,0.000000,0.000001,0.000001
3,3,1010 Micro-WKK Brandstofcel 2011 (WB bestaande bouw),Elektriciteit,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,4,1010 Micro-WKK Brandstofcel 2011 (WB bestaande bouw),Warmte,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



* Preview of 'Tech, Carrier & Sector by Year' (Book-EnergyBal with biofuels.xlsx) *



,Technology,Energy Carrier,Sector,2025,2030,2035,2040,2045,2050
0,1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw),Elektriciteit,Huishoudens,0.000000,-0.000000,-0.000000,0.000000,-0.000000,-0.000000
1,1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw),Warmte,Huishoudens,0.000000,-0.000000,-0.000000,0.000000,-0.000001,-0.000001
2,1003 Micro-WKK Brandstofcel 2011 (WB nieuwbouw),Waterstof,Huishoudens,0.000000,0.000000,0.000000,0.000000,0.000001,0.000001
3,1010 Micro-WKK Brandstofcel 2011 (WB bestaande bouw),Elektriciteit,Huishoudens,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,1010 Micro-WKK Brandstofcel 2011 (WB bestaande bouw),Warmte,Huishoudens,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [14]:

xl = pd.ExcelFile(file_path)
tech_ids = [
    "1012 Solar-PV Residential",
    "1042 Wind op Zee band 1",
    "1090 Wind op Land band 1",
    "1108 Natural gas CCGT",
    "1248 REF WKK gas Landbouw ebalans",
    "1462 Solar-PV Vertical (incl. bif noise barriers)",
    "1463 Solar-PV Bifacial vertical",
    "1464 Solar-PV Services",
    "1465 Solar-PV large scale utilities",
    "1497 STEG H2",
    "1583 Natural Gas CHP with CCS - Industry",
    "1599 Natural gas CCGT wCCS",
    "1721 REF AVI",
    "1756 Solar -PV industry",
    "1798 AVI CCS",
    "1852 REF Nuclear power plant",
    "1853 Nuclear energy Gen IV - Electricity production",
    "2023 Waste Gas CHP - Chemical",
    "2117 Solar PV floating > 1 MWp",
    "2127 Rooftop PV agriculture",
    "2220 REF Natural gas CHP - Low Temp - FBI",
    "2323 NG CHP - District heating",
    "2412 Nuclear energy EPR (Gen III+)  - Electricity production",
    "2511 Small Modular Reactor - electricity only modus",
    "2514 Small Modular Reactor - max heat modus"
]

# Filter the DataFrame for the specified technologies
df_filtered =combined_df[combined_df['Technology'].isin(tech_ids)]

# Pivot the DataFrame to have one column for each year, if not already in this format
# This step may need adjustment based on the actual structure of 'df_combined'
df_pivot = df_filtered.pivot_table(index=['Technology', 'Energy Carrier'],
                                    columns='Year', 
                                    values='Energy input/output', 
                                    fill_value=0).reset_index()

# Since the DataFrame is likely large, the pivot operation might need to be adjusted
# to fit your specific DataFrame structure and data types

# Write the filtered data to a new sheet in the Excel file

with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_pivot.to_excel(writer, sheet_name='Electricity & Heat', index=False)

print("Filtered data for specified technologies has been added to the 'Electricity & Heat' sheet.")


Filtered data for specified technologies has been added to the 'Electricity & Heat' sheet.


In [95]:
preview(file_path, "Electricity & Heat")


* Preview of 'Electricity & Heat' (Book-EnergyBal with biofuels.xlsx) *



,Technology,Energy Carrier,2025,2030,2035,2040,2045,2050
0,1012 Solar-PV Residential,Elektriciteit,-37.080362,-29.859767,-30.326279,-47.025141,-31.259398,-48.450141
1,1012 Solar-PV Residential,Zon,37.080362,29.859767,30.326279,47.025141,31.259398,48.450141
2,1042 Wind op Zee band 1,Elektriciteit,-81.121541,-188.900110,-232.982811,-232.766811,-306.865449,-307.307502
3,1042 Wind op Zee band 1,Wind,81.121542,188.900110,232.982811,232.766811,306.865449,307.307502
4,1090 Wind op Land band 1,Elektriciteit,-54.303554,-64.599915,-85.617193,-85.833189,-125.366056,-123.939430


In [17]:

sheet_name = "Tech, Carrier & Sector by Year"

#Public (Air and Water) transport technologies
#user can change these lists if their ESM uses different transport technologies
public_techs = [
    "1838 LNG inland navigation freight",
    "1995 Luchtvaart kerosine consumptie",
    "1995 Luchtvaart kerosine consumptie",
    "1995 Luchtvaart kerosine consumptie",
    "1996 Marine fuel vessel",
    "1996 Marine fuel vessel",
    "1996 Marine fuel vessel",
    "1996 Marine fuel vessel",
    "2383 Methanol vessel",
    "2383 Methanol vessel",
    "2383 Methanol vessel",
    "2385 LNG vessel",
    "2385 LNG vessel",
    "2385 LNG vessel",
    "2385 LNG vessel",
    "2385 LNG vessel",
    "2464 Ammonia vessel",
    "2431 REF ICE inland navigation freight",
    "2431 REF ICE inland navigation freight",
    "2431 REF ICE inland navigation freight",
    "2431 REF ICE inland navigation freight",
    "2432 ICE inland navigation freight - 15% fuel reduction",
    "2432 ICE inland navigation freight - 15% fuel reduction",
    "2432 ICE inland navigation freight - 15% fuel reduction",
    "2432 ICE inland navigation freight - 15% fuel reduction"
]

# Land transport technologies
private_techs = [
    "1365 LNG truck",
    "1648 Diesel truck with 5% fuel consumption reduction",
    "1648 Diesel truck with 5% fuel consumption reduction",
    "1648 Diesel truck with 5% fuel consumption reduction",
    "1650 Hybrid bus",
    "1650 Hybrid bus",
    "1650 Hybrid bus",
    "1651 REF Fuel cell bus",
    "1652 CNG bus",
    "1653 REF BEV bus",
    "1654 ICE hybrid diesel van",
    "1654 ICE hybrid diesel van",
    "1654 ICE hybrid diesel van",
    "1834 CNG van",
    "1835 H2 van",
    "1836 BEV van",
    "1837 PI van",
    "1837 PI van",
    "1837 PI van",
    "1837 PI van",
    "1841 REF ICE van",
    "1841 REF ICE van",
    "1841 REF ICE van",
    "1841 REF ICE van",
    "1841 REF ICE van",
    "1841 REF ICE van",
    "1841 REF ICE van",
    "1841 REF ICE van",
    "1841 REF ICE van",
    "1841 REF ICE van",
    "1841 REF ICE van",
    "2145 Electric truck with 600 km range",
    "2151 E batt truck ontladen",
    "2152 E batt truck opslag",
    "2153 E batt truck laden car park",
    "2154 E batt truck laden FS",
    "2371 LNG truck with 13% energy consumption reduction",
    "2372 H2 truck with energy consumption reduction",
    "2380 DME truck",
    "2381 DME truck with 18% energy consumption reduction",
    "2428 REF BEV van",
    "2429 REF electric truck",
    "2430 REF ICE bus",
    "2430 REF ICE bus",
    "2430 REF ICE bus",
    "2430 REF ICE bus"
]

# Load the main sheet 
df = pd.read_excel(file_path, sheet_name=sheet_name)

# Drop any unnamed or empty columns at the end
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
tech_col = df.columns[0]

# Helper function to extract all unique rows while preserving order
def extract_unique_tech_rows(df, tech_list):
    result = pd.DataFrame()
    seen = set()

    for tech in tech_list:
        matches = df[df[tech_col] == tech]
        for _, row in matches.iterrows():
            row_key = tuple(row[:3])  # Tech name, Energy Carrier, Sector
            if row_key not in seen:
                result = pd.concat([result, pd.DataFrame([row])], ignore_index=True)
                seen.add(row_key)
    return result

# Extract datasets
public_df = extract_unique_tech_rows(df, public_techs)
private_df = extract_unique_tech_rows(df, private_techs)

# Write to sheets
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    public_df.to_excel(writer, sheet_name="public transport", index=False)
    private_df.to_excel(writer, sheet_name="private transport", index=False)

print("Done: Extracted unique public & private transport rows with full energy carriers.")

print("Script 02 completed.")

Done: Extracted unique public & private transport rows with full energy carriers.


In [97]:
preview(file_path, "public transport")
preview(file_path, "private transport")


* Preview of 'public transport' (Book-EnergyBal with biofuels.xlsx) *



,Technology,Energy Carrier,Sector,2025,2030,2035,2040,2045,2050
0,1838 LNG inland navigation freight,Aardgas,Verkeer,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1995 Luchtvaart kerosine consumptie,Biokerosine,Bunkers,0.000000,6.800004,5.459681,26.868511,38.123877,61.046587
2,1995 Luchtvaart kerosine consumptie,Kerosine,Bunkers,29.042327,130.789965,139.820304,126.101496,121.678225,107.953413
3,1995 Luchtvaart kerosine consumptie,Synthetic kerosine,Bunkers,0.057664,0.100034,0.100020,0.100000,0.967903,1.000000
4,1996 Marine fuel vessel,BioHFO,Bunkers,0.000004,0.100021,0.100003,0.100000,0.100000,0.100006



* Preview of 'private transport' (Book-EnergyBal with biofuels.xlsx) *



,Technology,Energy Carrier,Sector,2025,2030,2035,2040,2045,2050
0,1365 LNG truck,Aardgas,Verkeer,0.000002,0.000030,0.000016,0.000000,0.000009,0.000017
1,1648 Diesel truck with 5% fuel consumption reduction,Biodiesel,Verkeer,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1648 Diesel truck with 5% fuel consumption reduction,Diesel,Verkeer,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,1648 Diesel truck with 5% fuel consumption reduction,Synthetic diesel,Verkeer,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,1650 Hybrid bus,Biodiesel,Verkeer,0.000000,0.000073,0.000083,0.000001,0.000019,0.000028
